# ML Training — CIC-IDS-2018 (Fixed Pipeline)

## What was wrong in the original notebook and what is fixed here:

| # | Problem | Fix Applied |
|---|---------|-------------|
| 1 | **Corrupt rows** — negative values in time/duration features (physically impossible) | Remove before anything else |
| 2 | **Duplicate rows** — 99.83% of Attack rows were identical, leaking into test set | `drop_duplicates()` before split |
| 3 | **Class imbalance** — 99.7% Attack, causing 0.99 accuracy by predicting one class | Undersample majority to 3:1 ratio |
| 4 | **Wrong metric** — plain accuracy is meaningless on imbalanced data | Use Balanced Accuracy + per-class Recall |
| 5 | **Scaling order** — confirmed data is raw, scaling is applied correctly (train only) | Split first → fit on train → transform both |

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import time
import pickle
import warnings
warnings.filterwarnings('ignore')
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PowerTransformer, StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report,
    balanced_accuracy_score
)

# ML Models
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import matplotlib.pyplot as plt
import seaborn as sns

print('=' * 80)
print('🚀 ML TRAINING — CIC-IDS-2018 FIXED PIPELINE')
print('=' * 80)

## 2. Configuration

In [ ]:
INPUT_FILE     = 'archive/full_df_binary_labels.csv'
BASE_MODEL_DIR = 'trained_models/ml_fixed/'
TEST_SIZE      = 0.2
RANDOM_STATE   = 42
SAVE_MODELS    = True

# Features that are physically impossible to be negative
# (time durations, packet counts, byte counts)
NON_NEGATIVE_FEATURES = [
    'Flow Duration', 'Flow IAT Mean', 'Flow IAT Std',
    'Flow IAT Max',  'Flow IAT Min',  'Fwd IAT Tot',
    'Fwd IAT Mean',  'Fwd IAT Std',   'Fwd IAT Max',
    'Fwd IAT Min',   'Bwd IAT Tot',   'Bwd IAT Mean',
    'Bwd IAT Std',   'Bwd IAT Max',   'Bwd IAT Min',
    'Idle Mean',     'Idle Max',      'Idle Min',
    'Tot Fwd Pkts',  'Tot Bwd Pkts',  'TotLen Fwd Pkts',
    'TotLen Bwd Pkts', 'Flow Byts/s', 'Flow Pkts/s'
]

# Features to apply PowerTransformer (extreme skew)
POWER_TRANSFORMER_FEATURES = [
    'Dst Port', 'Flow Duration', 'Tot Fwd Pkts', 'Tot Bwd Pkts',
    'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Fwd Pkt Len Max',
    'Fwd Pkt Len Mean', 'Fwd Pkt Len Std', 'Bwd Pkt Len Max',
    'Bwd Pkt Len Mean', 'Bwd Pkt Len Std', 'Flow Byts/s',
    'Flow Pkts/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max',
    'Flow IAT Min', 'Fwd IAT Tot', 'Fwd IAT Mean', 'Fwd IAT Std',
    'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Tot', 'Bwd IAT Mean',
    'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd Header Len',
    'Bwd Header Len', 'Fwd Pkts/s', 'Bwd Pkts/s', 'Pkt Len Max',
    'Pkt Len Mean', 'Pkt Len Std', 'Pkt Len Var', 'Down/Up Ratio',
    'Pkt Size Avg', 'Fwd Seg Size Avg', 'Bwd Seg Size Avg',
    'Subflow Fwd Pkts', 'Subflow Fwd Byts', 'Subflow Bwd Pkts',
    'Subflow Bwd Byts', 'Init Fwd Win Byts', 'Init Bwd Win Byts',
    'Fwd Act Data Pkts', 'Idle Mean', 'Idle Max', 'Idle Min'
]

# Features for StandardScaler (low skew)
STANDARD_SCALER_FEATURES = ['Fwd Seg Size Min']

# Features to leave unscaled (binary/categorical flags)
NO_SCALING_FEATURES = [
    'Protocol', 'RST Flag Cnt', 'PSH Flag Cnt',
    'ACK Flag Cnt', 'URG Flag Cnt', 'ECE Flag Cnt'
]

# Per-model sample sizes (None = use full balanced dataset)
MODEL_SAMPLE_SIZES = {
    'Random Forest':      None,
    'Decision Tree':      None,
    'Logistic Regression': None,
    'XGBoost':            None,
    'LightGBM':           None,
    'KNN':                100000,
    'Naive Bayes':        None,
    'SVM':                50000
}

os.makedirs(BASE_MODEL_DIR, exist_ok=True)
print('✅ Configuration set')
print(f'   Output directory: {BASE_MODEL_DIR}')

## 3. Load Dataset

In [ ]:
CHUNK_SIZE = 100000
MAX_ROWS   = None  # None = load all data

chunks     = []
total_rows = 0

print(f'\n📂 Loading data from: {INPUT_FILE}')
print('⏳ Reading in chunks...\n')

for i, chunk in enumerate(pd.read_csv(INPUT_FILE, chunksize=CHUNK_SIZE)):
    if MAX_ROWS and total_rows >= MAX_ROWS:
        break
    if MAX_ROWS and total_rows + len(chunk) > MAX_ROWS:
        chunk = chunk.head(MAX_ROWS - total_rows)
    chunks.append(chunk)
    total_rows += len(chunk)
    print(f'   Chunk {i+1}: {len(chunk):,} rows | Total: {total_rows:,}')

df_full = pd.concat(chunks, ignore_index=True)
del chunks

# Drop the string Label column — we only need Label_Binary
df_full = df_full.drop(columns=['Label'], errors='ignore')

print(f'\n✅ Dataset loaded')
print(f'   Rows   : {len(df_full):,}')
print(f'   Columns: {len(df_full.columns)}')
print(f'\nRaw class distribution:')
print(df_full['Label_Binary'].value_counts())

## 4. Data Cleaning
### Fix 1 — Remove corrupt rows (impossible negative values)
CICFlowMeter has a known bug where timestamp overflows produce **negative** values
in duration and inter-arrival time features. These rows are corrupted and must be removed
before any analysis or training.

In [ ]:
print('=' * 80)
print('🧹 FIX 1 — REMOVING CORRUPT ROWS (impossible negatives)')
print('=' * 80)

before = len(df_full)
cols_to_check = [c for c in NON_NEGATIVE_FEATURES if c in df_full.columns]

print('\nNegative value counts per feature:')
for col in cols_to_check:
    n_bad = (df_full[col] < 0).sum()
    if n_bad > 0:
        print(f'  ⚠️  {col:<30} {n_bad:>8,} corrupt values')

corrupt_mask = (df_full[cols_to_check] < 0).any(axis=1)
df_full = df_full[~corrupt_mask].copy()

removed = before - len(df_full)
print(f'\n  Before : {before:,}')
print(f'  Removed: {removed:,} corrupt rows')
print(f'  After  : {len(df_full):,}')

### Fix 2 — Remove duplicate rows
The diagnostic showed **99.83% of Attack rows are exact duplicates**.
When these are split into train/test, identical rows appear in both sets —
the model memorises them and gets perfect scores on rows it has already seen.

In [ ]:
print('=' * 80)
print('🧹 FIX 2 — REMOVING DUPLICATE ROWS')
print('=' * 80)

before = len(df_full)

# Check duplicates by class before removing
print('\nDuplicates by class (before removal):')
for label in df_full['Label_Binary'].unique():
    label_df  = df_full[df_full['Label_Binary'] == label]
    label_features = label_df.drop('Label_Binary', axis=1)
    n_dups    = label_features.duplicated().sum()
    pct       = n_dups / len(label_df) * 100
    name      = 'Benign' if label == 0 else 'Attack'
    print(f'  {name}: {n_dups:,} duplicates ({pct:.2f}%)')

df_full = df_full.drop_duplicates()

removed = before - len(df_full)
print(f'\n  Before : {before:,}')
print(f'  Removed: {removed:,} duplicates')
print(f'  After  : {len(df_full):,}')
print(f'\nClass distribution after dedup:')
print(df_full['Label_Binary'].value_counts())

### Fix 3 — Fix class imbalance
With 99%+ of rows being one class, any model that just predicts that class
scores 99% accuracy. We undersample the majority class to a **3:1 ratio**
so models are forced to actually learn both classes.

In [ ]:
print('=' * 80)
print('🧹 FIX 3 — FIXING CLASS IMBALANCE (undersampling to 3:1)')
print('=' * 80)

counts       = df_full['Label_Binary'].value_counts()
total        = len(df_full)
majority_pct = counts.max() / total * 100

print(f'\nBefore balancing:')
for label, count in counts.items():
    name = 'Benign (0)' if label == 0 else 'Attack (1)'
    print(f'  {name}: {count:,}  ({count/total*100:.2f}%)')
print(f'  Majority baseline accuracy: {majority_pct:.2f}%')

if majority_pct > 80:
    benign_df  = df_full[df_full['Label_Binary'] == 0]
    attack_df  = df_full[df_full['Label_Binary'] == 1]

    # Identify minority and majority
    if len(attack_df) > len(benign_df):
        minority_df = benign_df
        majority_df = attack_df
    else:
        minority_df = attack_df
        majority_df = benign_df

    # Keep all minority, sample majority to 3:1
    n_minority      = len(minority_df)
    n_majority_keep = min(len(majority_df), n_minority * 3)

    majority_sampled = majority_df.sample(n=n_majority_keep, random_state=RANDOM_STATE)
    df_full = pd.concat([minority_df, majority_sampled]) \
                .sample(frac=1, random_state=RANDOM_STATE) \
                .reset_index(drop=True)

    print(f'\nAfter balancing (3:1 ratio):')
    new_counts = df_full['Label_Binary'].value_counts()
    for label, count in new_counts.items():
        name = 'Benign (0)' if label == 0 else 'Attack (1)'
        print(f'  {name}: {count:,}  ({count/len(df_full)*100:.2f}%)')
    print(f'  Total rows: {len(df_full):,}')
else:
    print('  ✅ Class balance acceptable, no undersampling needed')

print('\n✅ Dataset is now clean and ready for training')

## 5. Prepare Features

In [ ]:
print('\n' + '=' * 80)
print('🔧 PREPARING FEATURES')
print('=' * 80)

X_full = df_full.drop(['Label_Binary'], axis=1).select_dtypes(include=[np.number])
y_full = df_full['Label_Binary']

# Remove constant features (zero variance — useless for any model)
constant_cols = [c for c in X_full.columns if X_full[c].std() < 1e-10]
if constant_cols:
    print(f'\n  Dropping {len(constant_cols)} constant features (zero variance):')
    for c in constant_cols:
        print(f'    - {c}')
    X_full = X_full.drop(columns=constant_cols)

del df_full

print(f'\n✅ Features shape : {X_full.shape}')
print(f'✅ Labels shape   : {y_full.shape}')
print(f'\nClass distribution:')
print(y_full.value_counts())

## 6. Helper Function — Prepare Data Per Model

In [ ]:
def get_model_data(X_full, y_full, model_name, sample_size=None,
                   test_size=0.2, random_state=42):
    """
    Prepare train/test data for a model.

    CORRECT ORDER:
      1. Sample (optional)
      2. Split train/test FIRST
      3. Fit scalers on TRAIN only
      4. Transform both sets with fitted scalers

    This prevents any form of data leakage.
    """
    X = X_full.copy()
    y = y_full.copy()

    # --- Optional sampling (for slow models like KNN, SVM) ---
    if sample_size is not None and sample_size < len(X):
        print(f'   📊 Sampling {sample_size:,} rows (from {len(X):,})')
        X, _, y, _ = train_test_split(
            X, y, train_size=sample_size,
            random_state=random_state, stratify=y
        )
        print(f'   Sampled class distribution: {y.value_counts().to_dict()}')
    else:
        print(f'   📊 Using full dataset: {len(X):,} rows')

    # --- STEP 1: Split FIRST — test set is isolated immediately ---
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        random_state=random_state,
        stratify=y
    )
    print(f'   Train: {len(X_train):,} rows | Test: {len(X_test):,} rows')

    X_train = X_train.copy()
    X_test  = X_test.copy()

    # --- STEP 2: Fit scalers on TRAIN only, transform both ---
    print('   🔧 Scaling (fit on train only — no leakage)...')
    scalers = {}

    # PowerTransformer — handles extreme skew and negative values
    power_cols = [
        c for c in POWER_TRANSFORMER_FEATURES
        if c in X_train.columns and X_train[c].std() > 1e-10
    ]
    if power_cols:
        pt = PowerTransformer(method='yeo-johnson', standardize=True)
        pt.fit(X_train[power_cols])                        # ← train only
        X_train[power_cols] = pt.transform(X_train[power_cols])
        X_test[power_cols]  = pt.transform(X_test[power_cols])
        scalers['power'] = pt
        print(f'      PowerTransformer : {len(power_cols)} features')

    # StandardScaler — for low-skew features
    std_cols = [
        c for c in STANDARD_SCALER_FEATURES
        if c in X_train.columns and X_train[c].std() > 1e-10
    ]
    if std_cols:
        ss = StandardScaler()
        ss.fit(X_train[std_cols])                          # ← train only
        X_train[std_cols] = ss.transform(X_train[std_cols])
        X_test[std_cols]  = ss.transform(X_test[std_cols])
        scalers['standard'] = ss
        print(f'      StandardScaler   : {len(std_cols)} features')

    no_scale = [c for c in NO_SCALING_FEATURES if c in X_train.columns]
    if no_scale:
        print(f'      Unscaled         : {len(no_scale)} binary/flag features')

    print('   ✅ Scaling complete')
    return X_train, X_test, y_train, y_test, scalers


print('✅ Helper function defined')

## 7. Define Models

In [ ]:
model_builders = {
    'Random Forest': lambda: RandomForestClassifier(
        n_estimators=100, max_depth=20, min_samples_split=10,
        n_jobs=-1, random_state=RANDOM_STATE
    ),
    'Decision Tree': lambda: DecisionTreeClassifier(
        max_depth=20, min_samples_split=10, random_state=RANDOM_STATE
    ),
    'Logistic Regression': lambda: LogisticRegression(
        max_iter=1000, n_jobs=-1, random_state=RANDOM_STATE
    ),
    'XGBoost': lambda: XGBClassifier(
        n_estimators=100, max_depth=10, learning_rate=0.1,
        n_jobs=-1, random_state=RANDOM_STATE, eval_metric='logloss'
    ),
    'LightGBM': lambda: LGBMClassifier(
        n_estimators=100, max_depth=10, learning_rate=0.1,
        n_jobs=-1, random_state=RANDOM_STATE, verbose=-1
    ),
    'KNN': lambda: KNeighborsClassifier(
        n_neighbors=5, n_jobs=-1
    ),
    'Naive Bayes': lambda: GaussianNB(),
    'SVM': lambda: SVC(
        kernel='rbf', C=1.0, random_state=RANDOM_STATE
    )
}

print(f'✅ Defined {len(model_builders)} models to train')

## 8. Train All Models

In [ ]:
results        = {}
trained_models = {}
saved_scalers  = {}

for model_name, model_builder in model_builders.items():
    print('\n' + '=' * 80)
    print(f'🚀 TRAINING: {model_name}')
    print('=' * 80)

    model_dir = os.path.join(BASE_MODEL_DIR, model_name.lower().replace(' ', '_'))
    os.makedirs(model_dir, exist_ok=True)

    sample_size = MODEL_SAMPLE_SIZES.get(model_name, None)

    # Prepare data — split first, scale on train only
    X_train, X_test, y_train, y_test, scalers = get_model_data(
        X_full, y_full, model_name, sample_size, TEST_SIZE, RANDOM_STATE
    )
    saved_scalers[model_name] = scalers

    # Train
    model = model_builder()
    print(f'\n⏱️  Training {model_name}...')
    start      = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start
    print(f'✅ Done in {train_time:.2f}s')

    # Evaluate
    y_pred = model.predict(X_test)

    acc      = accuracy_score(y_test, y_pred)
    bal_acc  = balanced_accuracy_score(y_test, y_pred)
    prec_w   = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec_w    = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1_w     = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    f1_macro = f1_score(y_test, y_pred, average='macro', zero_division=0)

    # Per-class metrics — the ones that actually matter
    report = classification_report(
        y_test, y_pred,
        target_names=['Benign', 'Attack'],
        output_dict=True, zero_division=0
    )
    benign_prec   = report['Benign']['precision']
    benign_recall = report['Benign']['recall']
    benign_f1     = report['Benign']['f1-score']
    attack_prec   = report['Attack']['precision']
    attack_recall = report['Attack']['recall']
    attack_f1     = report['Attack']['f1-score']

    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)

    print(f'\n📊 Results:')
    print(f'   Accuracy          : {acc:.4f}  (misleading on imbalanced data)')
    print(f'   Balanced Accuracy : {bal_acc:.4f}  ← PRIMARY METRIC')
    print(f'   F1 Macro          : {f1_macro:.4f}  ← USE THIS TOO')
    print(f'\n   Per-Class Performance:')
    print(f'   {"":20} {"Precision":>10} {"Recall":>10} {"F1":>10}')
    print(f'   {"-"*52}')
    print(f'   {"Benign":20} {benign_prec:>10.4f} {benign_recall:>10.4f} {benign_f1:>10.4f}')
    print(f'   {"Attack":20} {attack_prec:>10.4f} {attack_recall:>10.4f} {attack_f1:>10.4f}')
    print(f'\n   Confusion Matrix:')
    print(f'                      Pred Benign   Pred Attack')
    print(f'   Actual Benign      {cm[0][0]:>10,}   {cm[0][1]:>10,}')
    print(f'   Actual Attack      {cm[1][0]:>10,}   {cm[1][1]:>10,}')

    results[model_name] = {
        'accuracy':          acc,
        'balanced_accuracy': bal_acc,
        'precision_weighted': prec_w,
        'recall_weighted':   rec_w,
        'f1_weighted':       f1_w,
        'f1_macro':          f1_macro,
        'benign_precision':  benign_prec,
        'benign_recall':     benign_recall,
        'attack_precision':  attack_prec,
        'attack_recall':     attack_recall,
        'train_time':        train_time,
        'sample_size':       len(X_train)
    }

    # Save
    if SAVE_MODELS:
        with open(os.path.join(model_dir, 'model.pkl'), 'wb') as f:
            pickle.dump(model, f)
        if scalers:
            with open(os.path.join(model_dir, 'scalers.pkl'), 'wb') as f:
                pickle.dump(scalers, f)
        metadata = {
            'model_name':       model_name,
            'features':         X_train.columns.tolist(),
            'num_features':     len(X_train.columns),
            'sample_size':      len(X_train),
            'test_size':        len(X_test),
            'metrics':          results[model_name],
            'train_time':       train_time,
            'pipeline_fixes':   ['corrupt_rows_removed', 'duplicates_removed',
                                 'class_balanced_3to1', 'split_before_scale']
        }
        with open(os.path.join(model_dir, 'metadata.pkl'), 'wb') as f:
            pickle.dump(metadata, f)
        print(f'\n💾 Saved → {model_dir}')

    trained_models[model_name] = model

## 9. Results Summary

In [ ]:
print('\n' + '=' * 80)
print('📊 FINAL RESULTS — HONEST METRICS')
print('=' * 80)

summary_df = pd.DataFrame(results).T
summary_df = summary_df.sort_values('balanced_accuracy', ascending=False)

# Display the metrics that actually matter
display_cols = ['balanced_accuracy', 'f1_macro',
                'benign_recall', 'attack_recall',
                'benign_precision', 'attack_precision',
                'train_time', 'sample_size']

print('\nRanked by Balanced Accuracy (primary metric):')
print(summary_df[display_cols].to_string())

print('\n⚠️  NOTE: Plain accuracy is NOT shown — it is misleading on imbalanced data.')
print('    Balanced Accuracy treats both classes equally.')
print('    Benign Recall  = how many legitimate flows are correctly identified')
print('    Attack Recall  = how many attacks are correctly detected')

summary_df

## 10. Visualisation

In [ ]:
print('\n📊 Creating visualisations...')

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Model Performance — Fixed Pipeline (CIC-IDS-2018)',
             fontsize=16, fontweight='bold')

model_names = summary_df.index.tolist()

# Balanced Accuracy
axes[0, 0].barh(model_names, summary_df['balanced_accuracy'], color='steelblue')
axes[0, 0].set_xlabel('Balanced Accuracy')
axes[0, 0].set_title('Balanced Accuracy (Primary Metric)', fontweight='bold')
axes[0, 0].set_xlim(0, 1)
axes[0, 0].axvline(0.5, color='red', linestyle='--', alpha=0.5, label='Random baseline')
axes[0, 0].legend(fontsize=8)

# F1 Macro
axes[0, 1].barh(model_names, summary_df['f1_macro'], color='coral')
axes[0, 1].set_xlabel('F1 Macro')
axes[0, 1].set_title('F1 Macro Score', fontweight='bold')
axes[0, 1].set_xlim(0, 1)

# Per-class Recall comparison
x      = np.arange(len(model_names))
width  = 0.35
axes[1, 0].bar(x - width/2, summary_df['benign_recall'],  width, label='Benign Recall',  color='steelblue')
axes[1, 0].bar(x + width/2, summary_df['attack_recall'],  width, label='Attack Recall',  color='coral')
axes[1, 0].set_xlabel('Model')
axes[1, 0].set_ylabel('Recall')
axes[1, 0].set_title('Per-Class Recall (Most Important)', fontweight='bold')
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels(model_names, rotation=45, ha='right')
axes[1, 0].set_ylim(0, 1)
axes[1, 0].legend()

# Training time
axes[1, 1].barh(model_names, summary_df['train_time'], color='lightgreen')
axes[1, 1].set_xlabel('Training Time (seconds)')
axes[1, 1].set_title('Training Time', fontweight='bold')

plt.tight_layout()
plot_path = os.path.join(BASE_MODEL_DIR, 'model_comparison_fixed.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
print(f'✅ Saved → {plot_path}')
plt.show()

## 11. Confusion Matrices

In [ ]:
n_models = len(trained_models)
ncols    = 4
nrows    = (n_models + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4, nrows * 3.5))
axes = axes.flatten()

for idx, (model_name, model) in enumerate(trained_models.items()):
    # Re-generate predictions for confusion matrix
    sample_size = MODEL_SAMPLE_SIZES.get(model_name, None)
    X_tr, X_te, y_tr, y_te, _ = get_model_data(
        X_full, y_full, model_name, sample_size, TEST_SIZE, RANDOM_STATE
    )
    y_pred = model.predict(X_te)
    cm     = confusion_matrix(y_te, y_pred)

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=['Benign', 'Attack'],
                yticklabels=['Benign', 'Attack'])
    bal = balanced_accuracy_score(y_te, y_pred)
    axes[idx].set_title(f'{model_name}\nBal.Acc={bal:.3f}', fontweight='bold', fontsize=9)
    axes[idx].set_ylabel('Actual')
    axes[idx].set_xlabel('Predicted')

# Hide unused subplots
for idx in range(len(trained_models), len(axes)):
    axes[idx].set_visible(False)

plt.suptitle('Confusion Matrices — All Models', fontsize=14, fontweight='bold')
plt.tight_layout()
cm_path = os.path.join(BASE_MODEL_DIR, 'confusion_matrices_fixed.png')
plt.savefig(cm_path, dpi=150, bbox_inches='tight')
print(f'✅ Saved → {cm_path}')
plt.show()

## 12. Final Summary

In [ ]:
print('\n' + '=' * 80)
print('🎉 TRAINING COMPLETE — FIXED PIPELINE')
print('=' * 80)

print('\n📋 What was fixed in this notebook:')
print('   ✅ Corrupt rows removed  (impossible negative values from CICFlowMeter bug)')
print('   ✅ Duplicates removed    (99.83% of Attack rows were identical)')
print('   ✅ Class imbalance fixed (undersampled to 3:1 Attack:Benign ratio)')
print('   ✅ Scaling done correctly (split first → fit on train only)')
print('   ✅ Honest metrics used   (Balanced Accuracy + per-class Recall)')

print('\n📊 Models Ranked by Balanced Accuracy:')
for rank, (model_name, row) in enumerate(
        sorted(results.items(),
               key=lambda x: x[1]['balanced_accuracy'],
               reverse=True), 1):
    print(f'   {rank}. {model_name:<22}  '
          f'Bal.Acc={row["balanced_accuracy"]:.4f}  '
          f'Benign Recall={row["benign_recall"]:.4f}  '
          f'Attack Recall={row["attack_recall"]:.4f}')

print(f'\n📁 All models saved to: {BASE_MODEL_DIR}')
print('\n💡 Key reminder:')
print('   A good IDS model needs HIGH recall on BOTH classes.')
print('   Low Benign Recall → too many false alarms.')
print('   Low Attack Recall → attacks are slipping through.')